# Network Security Scanner and Vulnerability Assessment Tool

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import socket
import threading
import subprocess
import requests
import hashlib
import os
import time
import json
from datetime import datetime
import ipaddress
import re

class NetworkScanner:
    def __init__(self):
        self.open_ports = []
        self.vulnerabilities = []
        self.scan_results = {}

    def port_scan(self, target, port):
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex((target, port))
            if result == 0:
                self.open_ports.append(port)
                try:
                    banner = sock.recv(1024).decode().strip()
                    self.scan_results[port] = banner
                except:
                    self.scan_results[port] = "No banner"
            sock.close()
        except:
            pass

    def threaded_scan(self, target, ports):
        threads = []
        for port in ports:
            thread = threading.Thread(target=self.port_scan, args=(target, port))
            threads.append(thread)
            thread.start()

        for thread in threads:
            thread.join()

    def vulnerability_check(self, target, port):
        vulns = []
        if port == 21:
            vulns.append("FTP Anonymous login possible")
        elif port == 23:
            vulns.append("Telnet - Unencrypted protocol")
        elif port == 80:
            vulns.append("HTTP - Check for web vulnerabilities")
        elif port == 443:
            vulns.append("HTTPS - Check SSL/TLS configuration")
        elif port == 3389:
            vulns.append("RDP - Remote access point")
        elif port == 22:
            vulns.append("SSH - Check for weak credentials")
        return vulns

class WebVulnScanner:
    def __init__(self):
        self.session = requests.Session()
        self.vulnerabilities = []

    def sql_injection_test(self, url):
        payloads = ["'", "1' OR '1'='1", "1; DROP TABLE users--", "1' UNION SELECT NULL--"]
        for payload in payloads:
            try:
                response = self.session.get(f"{url}?id={payload}", timeout=5)
                if any(error in response.text.lower() for error in ['sql', 'mysql', 'error', 'warning']):
                    self.vulnerabilities.append(f"Potential SQL Injection: {url}")
                    break
            except:
                continue

    def xss_test(self, url):
        payload = "<script>alert('XSS')</script>"
        try:
            response = self.session.get(f"{url}?q={payload}", timeout=5)
            if payload in response.text:
                self.vulnerabilities.append(f"Potential XSS: {url}")
        except:
            pass

    def directory_traversal_test(self, url):
        payloads = ["../../../etc/passwd", "..\\..\\..\\windows\\system32\\drivers\\etc\\hosts"]
        for payload in payloads:
            try:
                response = self.session.get(f"{url}?file={payload}", timeout=5)
                if "root:" in response.text or "localhost" in response.text:
                    self.vulnerabilities.append(f"Directory Traversal: {url}")
                    break
            except:
                continue

class PasswordAnalyzer:
    def __init__(self):
        self.common_passwords = [
            "123456", "password", "123456789", "12345678", "12345",
            "1234567", "1234567890", "qwerty", "abc123", "111111"
        ]

    def strength_check(self, password):
        score = 0
        feedback = []

        if len(password) >= 8:
            score += 1
        else:
            feedback.append("Use at least 8 characters")

        if re.search(r'[A-Z]', password):
            score += 1
        else:
            feedback.append("Add uppercase letters")

        if re.search(r'[a-z]', password):
            score += 1
        else:
            feedback.append("Add lowercase letters")

        if re.search(r'\d', password):
            score += 1
        else:
            feedback.append("Add numbers")

        if re.search(r'[!@#$%^&*(),.?":{}|<>]', password):
            score += 1
        else:
            feedback.append("Add special characters")

        if password.lower() in self.common_passwords:
            score = 0
            feedback = ["Password too common - choose something unique"]

        strength_levels = ["Very Weak", "Weak", "Fair", "Good", "Strong"]
        return strength_levels[min(score, 4)], feedback

    def generate_hash(self, password, algorithm='sha256'):
        if algorithm == 'md5':
            return hashlib.md5(password.encode()).hexdigest()
        elif algorithm == 'sha1':
            return hashlib.sha1(password.encode()).hexdigest()
        elif algorithm == 'sha256':
            return hashlib.sha256(password.encode()).hexdigest()
        else:
            return hashlib.sha512(password.encode()).hexdigest()

class SecurityAuditor:
    def __init__(self):
        self.findings = []
        self.network_scanner = NetworkScanner()
        self.web_scanner = WebVulnScanner()
        self.password_analyzer = PasswordAnalyzer()

    def network_audit(self, target_ip, port_range=None):
        if port_range is None:
            port_range = [21, 22, 23, 25, 53, 80, 110, 143, 443, 993, 995, 3389, 5432, 3306]

        print(f"Scanning {target_ip}...")
        self.network_scanner.threaded_scan(target_ip, port_range)

        results = {
            'target': target_ip,
            'open_ports': self.network_scanner.open_ports,
            'banners': self.network_scanner.scan_results,
            'vulnerabilities': []
        }

        for port in self.network_scanner.open_ports:
            vulns = self.network_scanner.vulnerability_check(target_ip, port)
            results['vulnerabilities'].extend(vulns)

        return results

    def web_audit(self, target_url):
        print(f"Testing web vulnerabilities on {target_url}...")
        self.web_scanner.sql_injection_test(target_url)
        self.web_scanner.xss_test(target_url)
        self.web_scanner.directory_traversal_test(target_url)

        return {
            'target': target_url,
            'vulnerabilities': self.web_scanner.vulnerabilities
        }

    def password_audit(self, passwords):
        results = []
        for password in passwords:
            strength, feedback = self.password_analyzer.strength_check(password)
            hash_md5 = self.password_analyzer.generate_hash(password, 'md5')
            hash_sha256 = self.password_analyzer.generate_hash(password, 'sha256')

            results.append({
                'password': '*' * len(password),
                'strength': strength,
                'feedback': feedback,
                'md5_hash': hash_md5,
                'sha256_hash': hash_sha256
            })

        return results

    def generate_report(self, audit_data):
        report = {
            'timestamp': datetime.now().isoformat(),
            'audit_results': audit_data,
            'summary': {
                'total_findings': len(audit_data.get('vulnerabilities', [])),
                'risk_level': self.calculate_risk_level(audit_data)
            }
        }
        return json.dumps(report, indent=2)

    def calculate_risk_level(self, data):
        high_risk_ports = [21, 23, 3389]
        open_high_risk = any(port in data.get('open_ports', []) for port in high_risk_ports)
        vuln_count = len(data.get('vulnerabilities', []))

        if open_high_risk or vuln_count > 5:
            return "HIGH"
        elif vuln_count > 2:
            return "MEDIUM"
        else:
            return "LOW"

def demo_scan():
    auditor = SecurityAuditor()

    print("=== CYBERSECURITY AUDIT DEMO ===\n")

    print("1. Network Scanning Demo")
    try:
        results = auditor.network_audit("127.0.0.1", [22, 80, 443, 8080])
        print(f"Open ports: {results['open_ports']}")
        print(f"Potential issues: {len(results['vulnerabilities'])}")
    except Exception as e:
        print(f"Network scan error: {e}")

    print("\n2. Password Analysis Demo")
    test_passwords = ["123456", "MySecureP@ssw0rd!", "abc123", "Str0ng!P@ssw0rd2024"]
    password_results = auditor.password_audit(test_passwords)

    for result in password_results:
        print(f"Password strength: {result['strength']}")
        if result['feedback']:
            print(f"Recommendations: {', '.join(result['feedback'])}")

    print("\n3. Web Vulnerability Test (Demo URLs)")
    try:
        web_results = auditor.web_audit("http://testphp.vulnweb.com")
        print(f"Web vulnerabilities found: {len(web_results['vulnerabilities'])}")
    except:
        print("Web scan demo unavailable")

    print("\n=== SCAN COMPLETE ===")

if __name__ == "__main__":
    demo_scan()